In this notebook we seek to cluster the images based en their similarity metrics computed in test_code.dfield_approach_elastix.approach_images_similar

In [31]:
import os
import sys

current_path = os.path.abspath(os.getcwd())
base_path = os.path.abspath(os.path.join(current_path, '../../../'))
sys.path.append(base_path)

import numpy as np
import matplotlib.pyplot as plt  
import pandas as pd
from IPython.display import display


import project_images_loader as pil
import utils.util as util
import utils.functions as fc
import utils.nifti_functions as nfc

from test_code.dfield_approach_elastix.approach_images_similar.similarity_tables_reader import SimilarityTablesReader

from sklearn.cluster import DBSCAN
np.set_printoptions(precision=4)



## Read similarity table

In [32]:
matching_ids = pil.get_matching_ids()
sim_reader = SimilarityTablesReader(matching_ids)
sim_table = sim_reader.get_all_images_similarity_table("dice_mean")
sim_table_np = sim_table.values
np.fill_diagonal(sim_table_np, 1)

def apply_non_linear_distance_tranformation(sim_table_np, type="exp", alpha=1.0):
    if type == "lineal":
        sim_table_np = alpha * (1 - sim_table_np)
    elif type == "inv":
        sim_table_np = 1 / (alpha * sim_table_np)
    elif type == "exp":
        sim_table_np = 1 - sim_table_np
        sim_table_np = np.exp(alpha * sim_table_np)
    return sim_table_np

# for type in ["lineal", "inv", "exp"]:
#     for alpha in [0.1, 1.0, 10.0]:
#         _sim_table_np = apply_non_linear_distance_tranformation(sim_table_np, type=type, alpha=alpha)
#         print(f"Transformation: {type} - alpha: {alpha}")
#         print(f"min: {np.min(_sim_table_np)}\t max: {np.max(_sim_table_np)}\t mean: {np.mean(_sim_table_np)}")
#         print(_sim_table_np[:5, :5])
#         print("\n\n")


## Find best clustering

In [33]:
def find_best_parameters(sim_table_np):
    best_cluster_list = []
    nb_bins = 1000

    min_table = np.min(sim_table_np)
    eps_space = np.linspace(min_table, np.max(sim_table_np), nb_bins)

    for eps in eps_space:
        if (eps == min_table):
            continue
        for min_samples in range(1, 20):
            dbscan_obj = DBSCAN(eps=eps, min_samples=min_samples, metric='precomputed').fit(sim_table_np)
            nb_clusters = len(np.unique(dbscan_obj.labels_))
            # print(f"{i}: {nb_clusters}")
            if (nb_clusters > 1 and nb_clusters < 20):
                nb_noise = len(np.where(dbscan_obj.labels_ == -1)[0])
                to_minimize = 1/nb_clusters + nb_noise

                best_cluster_list.append((eps, min_samples, nb_noise, nb_clusters, to_minimize))
            # print(f"e: {e}\t min_samples: {min_samples}\t nb_noise: {nb_noise}\t nb_clusters: {nb_clusters}")

    best_cluster_list = np.array(best_cluster_list)
    sorted_indices = np.argsort(best_cluster_list[:, -1])
    sorted_best_cluster_list = best_cluster_list[sorted_indices]

    print(sorted_best_cluster_list[:10])
    return sorted_best_cluster_list

_sim_table_np = apply_non_linear_distance_tranformation(sim_table_np, type="exp", alpha=2.0)
sorted_best_cluster_list = find_best_parameters(_sim_table_np)


[[ 3.2784  1.      0.     19.      0.0526]
 [ 3.2844  1.      0.     19.      0.0526]
 [ 3.2904  1.      0.     17.      0.0588]
 [ 3.3025  1.      0.     16.      0.0625]
 [ 3.2965  1.      0.     16.      0.0625]
 [ 3.3086  1.      0.     16.      0.0625]
 [ 3.3146  1.      0.     15.      0.0667]
 [ 3.3207  1.      0.     14.      0.0714]
 [ 3.3267  1.      0.     14.      0.0714]
 [ 3.3327  1.      0.     13.      0.0769]]


In [35]:
def find_list_with_more_elements_and_min_std(listas):
    longitudes = [(i, len(lista)) for i, lista in enumerate(listas)]
    longitudes.sort(key=lambda x: x[1], reverse=True)
    mejor_lista_index_list = min(longitudes, key=lambda x: np.std(listas[x[0]]))
    return mejor_lista_index_list[0]

def ordenar_listas_por_std(listas):
    stds = [np.std(lista) for lista in listas]
    sorted_listas = np.argsort(stds)
    return sorted_listas


def compute_clusters_list(sorted_best_cluster_list, _sim_table_np):
    clusters_list = []
    for row in sorted_best_cluster_list:
        eps, min_samples, nb_noise, nb_clusters, to_minimize = row
        if nb_noise >-1:
            dbscan_obj = DBSCAN(eps=eps, min_samples=int(min_samples), metric='precomputed')
            dbscan_obj.fit(_sim_table_np)
            clusters_list.append(np.unique(dbscan_obj.labels_, return_counts=True)[1])
    return clusters_list

clusters_list = compute_clusters_list(sorted_best_cluster_list, _sim_table_np)
best_index = find_list_with_more_elements_and_min_std(clusters_list)
sorted_std_index = ordenar_listas_por_std(clusters_list)
print(f"Best index: {best_index}, clusters: {clusters_list[best_index]}")
print(sorted_best_cluster_list[best_index])



Best index: 5881, clusters: [145 163]
[  3.0427   6.     145.       2.     145.5   ]


In [36]:
print(sorted_std_index)

[5881 5872 5865 ...  770  782  809]


In [44]:
for index in sorted_std_index:
    eps, min_samples, nb_noise, nb_clusters, to_minimize = sorted_best_cluster_list[index]
    if nb_clusters > 2 and nb_clusters < 11:
        dbscan_obj = DBSCAN(eps=eps, min_samples=int(min_samples), metric='precomputed')
        dbscan_obj.fit(_sim_table_np)
        print(f"best eps: {eps}\t nb_clusters: {nb_clusters}")
        print(np.unique(dbscan_obj.labels_, return_counts=True)[1])
        print(dbscan_obj.labels_)




best eps: 3.1876999363781824	 nb_clusters: 3.0
[ 96  87 125]
[ 0  1 -1  0  0  1 -1  0  0 -1  0 -1 -1  1  1  1  1  0  0 -1  0  1  0  1
 -1 -1  1  0  1 -1  1  0  0  0  0  1  0  1  0 -1 -1  0 -1 -1  0  1  1  1
 -1  1  1  1  0 -1  1  1 -1  0 -1  0  1  1  0 -1  0  1  0  1  1 -1 -1  0
 -1  0 -1  0 -1  1  1  1  0  1  1 -1 -1  0  1  0  0 -1  0 -1  1  0  1  1
  1 -1  0 -1  1  1  1  1 -1 -1  1 -1 -1  1 -1  1 -1 -1  1  1  0 -1  1 -1
  0  0  0  0  1 -1 -1 -1  0  0  0 -1 -1 -1  0  1 -1 -1  0  0  1 -1  1  1
 -1  0  0  0  1  1 -1 -1  1  1  1  1  1  1  0  1  1  1 -1  0  1  0 -1 -1
  0  0  0 -1  0  1  1 -1  0  0  1 -1  0  0  1 -1  1  0 -1  0 -1  1  0  0
  0  0 -1 -1 -1  1  1  1  1 -1 -1  0  1  1 -1  0  1 -1  0 -1  1  1  0  0
  1 -1  1 -1 -1  1 -1  1  0  1  1  1 -1  1  1  1 -1  1 -1  1  1 -1  1  1
  0  1 -1  1 -1 -1  0  0  1  1  1  0  1  1  0  0 -1  1 -1  0  1  0  1 -1
 -1 -1  1 -1  1 -1 -1  0 -1  1  1  1 -1  1  1  1  0  1  1  1  1 -1  0 -1
 -1  1 -1  1  1 -1  0  1  0  1  1  0  1 -1  0 -1  1 -1  1  1]
b

In [21]:
eps, min_samples, nb_noise, nb_clusters, to_minimize = sorted_best_cluster_list[best_index]
dbscan_obj = DBSCAN(eps=eps, min_samples=int(min_samples), metric='precomputed')
dbscan_obj.fit(_sim_table_np)
nb_clusters = len(np.unique(dbscan_obj.labels_))
print(f"best eps: {eps}\t nb_clusters: {nb_clusters}")
print(np.unique(dbscan_obj.labels_, return_counts=True)[1])
print(dbscan_obj.labels_)

best eps: 3.042659056618303	 nb_clusters: 2
[145 163]
[ 0  0 -1  0 -1  0 -1 -1 -1 -1  0 -1 -1  0  0  0  0  0  0 -1  0  0  0  0
 -1 -1  0  0  0 -1  0  0  0  0 -1 -1 -1  0 -1 -1  0  0 -1 -1  0  0 -1 -1
 -1 -1  0  0  0 -1  0  0 -1  0 -1  0 -1  0  0 -1  0  0  0  0  0 -1  0  0
 -1  0 -1  0 -1  0 -1  0  0  0  0 -1 -1  0 -1  0  0 -1  0 -1  0 -1  0  0
  0 -1  0 -1  0  0  0  0 -1 -1  0 -1 -1 -1  0  0 -1 -1 -1  0 -1 -1  0 -1
  0 -1  0  0  0 -1 -1 -1  0  0  0 -1 -1 -1 -1  0 -1 -1  0  0 -1 -1  0 -1
 -1 -1  0  0  0 -1 -1 -1  0  0  0 -1  0  0  0 -1  0  0 -1 -1  0 -1 -1 -1
  0  0  0 -1  0  0 -1 -1  0 -1  0 -1 -1  0  0 -1  0  0 -1  0 -1 -1  0 -1
  0  0 -1 -1 -1  0 -1  0 -1 -1 -1 -1  0  0 -1  0 -1  0  0 -1  0 -1  0 -1
  0 -1  0 -1 -1 -1 -1  0  0  0  0  0 -1  0  0  0 -1  0 -1  0  0 -1  0  0
  0  0 -1 -1 -1 -1  0  0 -1  0  0  0 -1 -1  0  0 -1  0  0  0  0  0  0 -1
 -1 -1  0 -1 -1 -1 -1 -1 -1  0  0  0 -1 -1  0 -1 -1  0 -1  0  0 -1 -1 -1
 -1  0 -1 -1  0 -1 -1  0  0  0  0  0 -1 -1  0 -1 -1 -1  0  0]
